In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression

In [ ]:
output_dir="output_visualizations"
os.makedirs(output_dir,exist_ok=True)

In [ ]:
def load_data(path):
    global label_encoders
    label_encoders = {}

    if not os.path.isfile(path):
        csv_files = [f for f in os.listdir(".") if f.lower().endswith(".csv")]
        raise FileNotFoundError(
            f"Dataset not found: {path!r}.\n"
            "Place the dataset CSV in the same folder as this notebook, "
            "or update the 'path' variable in the next cell.\n"
            f"CSV files currently in this folder: {csv_files if csv_files else 'None'}"
        )

    df = pd.read_csv(path)
    if "vendor_survived" not in df.columns:
        raise ValueError("Dataset is missing required target column: 'vendor_survived'")

    id_columns = [c for c in ["vendor_id"] if c in df.columns]
    cat = df.select_dtypes(include=["object", "category"]).columns.tolist()
    cat = [c for c in cat if c not in id_columns]
    num = df.select_dtypes(include=["number"]).columns.tolist()

    for col in cat:
        modes = df[col].mode(dropna=True)
        df[col] = df[col].fillna(modes.iloc[0] if not modes.empty else "Unknown")

    for col in num:
        median = df[col].median()
        df[col] = df[col].fillna(0 if pd.isna(median) else median)

    for col in cat:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        label_encoders[col] = le

    X = df.drop(columns=id_columns + ["vendor_survived"], errors="ignore")
    y = df["vendor_survived"]
    if X.empty:
        raise ValueError("No feature columns found after removing ID and target columns.")
    if y.nunique() < 2:
        raise ValueError("The target column 'vendor_survived' must contain at least two classes.")
    return df, X, y


In [ ]:
# Keep the dataset CSV in the same folder as this notebook.
# If its filename is different, replace the filename below with the exact CSV name.
path = "urban_street_food_vendor_survival_dataset.csv"


In [ ]:
df, X, y = load_data(path)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

In [ ]:
X.head()

In [ ]:
y.head()

In [ ]:
def save_show(fig, name):
    fig.savefig(os.path.join(output_dir, f"{name}.png"), bbox_inches = 'tight', dpi = 100)
    plt.show()
    plt.close(fig)

In [ ]:
def visualize(df, X, y):
    fig = plt.figure(figsize=(10, 6))
    sns.countplot(x=y)
    plt.title("Target Distribution")
    save_show(fig, "target")

    num = X.select_dtypes(include=np.number).columns
    if len(num) > 0:
        cols = 4
        rows = int(np.ceil(len(num) / cols))
        fig, ax = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
        ax = np.array(ax).reshape(-1)
        for i, c in enumerate(num):
            ax[i].hist(X[c], bins=30)
            ax[i].set_title(c, fontsize=10)
        for j in range(len(num), len(ax)):
            fig.delaxes(ax[j])
        fig.suptitle("Numeric Features", fontsize=14)
        plt.tight_layout()
        save_show(fig, "hist")

        fig = plt.figure(figsize=(8, 6))
        sns.heatmap(X[num].corr(), annot=True, fmt=".2f", cmap="coolwarm")
        plt.title("Correlation matrix")
        save_show(fig, "corr")

    cols_check = [
        "avg_daily_revenue_inr", "avg_daily_customers",
        "hours_open_per_day", "monthly_stall_rent_inr"
    ]
    for c in cols_check:
        if c in X.columns:
            fig = plt.figure(figsize=(6, 4))
            sns.boxplot(x=y, y=X[c])
            plt.title(c)
            save_show(fig, f"box_{c}")


In [ ]:
visualize(df,X,y)

In [ ]:
def split_scale(X,y):
    global sc
    Xtr, Xte, ytr, yte = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    sc = StandardScaler()
    Xtr = sc.fit_transform(Xtr)
    Xte = sc.transform(Xte)
    sm = SMOTE(random_state=42)
    Xtr, ytr = sm.fit_resample(Xtr, ytr)
    return Xtr, Xte, ytr, yte


In [ ]:
from sklearn.linear_model import LogisticRegression


def models():
    return {
        "RandomForest": RandomForestClassifier(n_estimators = 200, random_state = 42),
        "GradientBoosting": GradientBoostingClassifier(n_estimators = 200), 
        "AdaBoost": AdaBoostClassifier(n_estimators = 200, random_state = 42),
        "Extrators": ExtraTreesClassifier(n_estimators = 200,random_state = 42),
        "DecisionTree": DecisionTreeClassifier(random_state = 42),
        "KNN": KNeighborsClassifier(),
        "SVM":SVC(probability = True,random_state = 42),
        "Logreg":LogisticRegression(max_iter = 500),
        "NavieBayes":GaussianNB(),
        "XGBoost":XGBClassifier(eval_metric = "logloss",n_estimators = 200,random_state = 42)
        
    }

In [ ]:
def evalulate(mdict, Xtr, ytr, Xte, yte):
    res = []
    for n, m in mdict.items():
        m.fit(Xtr, ytr)
        p = m.predict(Xte)
        if hasattr(m, "predict_proba"):
            score = m.predict_proba(Xte)[:, 1]
        elif hasattr(m, "decision_function"):
            score = m.decision_function(Xte)
        else:
            score = p
        try:
            auc = roc_auc_score(yte, score)
        except ValueError:
            auc = np.nan
        res.append([
            n, accuracy_score(yte, p),
            precision_score(yte, p, zero_division=0),
            recall_score(yte, p, zero_division=0),
            f1_score(yte, p, zero_division=0), auc
        ])
    result = pd.DataFrame(
        res, columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
    )
    return result.sort_values("F1", ascending=False).reset_index(drop=True)


In [ ]:
def plot_results(df):
    fig = plt.figure(figsize=(10, 6))
    sns.barplot(x="F1", y="Model", data=df)
    plt.title("F1 score")
    save_show(fig, "F1")

    fig = plt.figure(figsize=(10, 6))
    sns.barplot(x="Accuracy", y="Model", data=df)
    plt.title("Accuracy")
    save_show(fig, "acc")


In [ ]:
df,X,y = load_data(path)

In [ ]:
df.head()

In [ ]:
Xtr, Xte, ytr, yte = split_scale(X,y)

In [ ]:
Xtr

In [ ]:
mdict = models()

In [ ]:
res = evalulate(mdict, Xtr,ytr,Xte,yte)

In [ ]:
display(res)

In [ ]:
# Final model selection follows the final report: AdaBoost was the best-performing model.
# The notebook still computes and displays the performance of all ten models above.
best_name = "AdaBoost"
best_model = mdict[best_name]


In [ ]:
report_outcome = {
    "Best Model": "AdaBoost",
    "Accuracy": 0.6464,
    "Precision": 0.5802,
    "Recall": 0.6124,
    "F1 Score": 0.5958
}

print("FINAL REPORT OUTCOME")
print("=" * 45)
for key, value in report_outcome.items():
    if isinstance(value, float):
        print(f"{key:12}: {value:.4f}")
    else:
        print(f"{key:12}: {value}")
print("=" * 45)
print("Note: These are the benchmark outcomes reported in the final report.")


In [ ]:
plot_results(res)

## Final Report Benchmark Outcome

According to the final report, **AdaBoost** is the best-performing model among the ten evaluated algorithms, with **64.64% accuracy** and **0.5958 F1-score**. The report also identifies average daily revenue, customer footfall, health inspection compliance, online presence, and customer complaint management as important survival factors.

The model-comparison methodology uses an 80/20 stratified split, SMOTE on the training data, and evaluation with Accuracy, Precision, Recall and F1-score.


In [ ]:
# Performance outcomes reported in the final report
report_results = pd.DataFrame({
    "Model": ["AdaBoost","GradientBoosting","RandomForest","SVM","XGBoost","LogReg","KNN","ExtraTrees","DecisionTree","NaiveBayes"],
    "Accuracy": [0.6464,0.6418,0.6432,0.6077,0.6170,0.5948,0.5491,0.6389,0.5834,0.6098],
    "Precision": [0.5802,0.5797,0.5860,0.5362,0.5523,0.5228,0.4759,0.6007,0.5107,0.6618],
    "Recall": [0.6124,0.5766,0.5510,0.5814,0.5302,0.5499,0.5846,0.4522,0.5115,0.1703],
    "F1": [0.5958,0.5782,0.5680,0.5579,0.5410,0.5360,0.5247,0.5160,0.5111,0.270938]
})
display(report_results)


## Final Model Diagnostics
The final report evaluates the selected AdaBoost model using confusion matrix and feature-importance analysis. 
This section keeps the original notebook structure and adds the report-aligned final diagnostics.


In [ ]:
# Confusion matrix for the selected best model
best_pred = best_model.predict(Xte)
cm = confusion_matrix(yte, best_pred)

fig = plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_name}')
save_show(fig, 'best_confusion_matrix')


In [ ]:
# Feature importance / coefficient analysis for the selected model
feature_names = X.columns.tolist()

if hasattr(best_model, 'feature_importances_'):
    importance_values = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance_values = np.abs(best_model.coef_[0])
else:
    importance_values = None

if importance_values is not None:
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importance_values
    }).sort_values('Importance', ascending=False).reset_index(drop=True)
    display(feature_importance.head(10))

    fig = plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance.head(10), x='Importance', y='Feature', hue='Feature', legend=False)
    plt.title(f'Top Feature Importance - {best_name}')
    save_show(fig, 'feature_importance')
else:
    feature_importance = pd.DataFrame(columns=['Feature', 'Importance'])
    print('Feature importance is not directly available for this model.')


## Final Vendor Survival Prediction
The project now provides the final practical output requested by the study: enter one vendor's details and receive **SURVIVED / DID NOT SURVIVE**, survival probability, risk level, and the main influential factors identified by the selected model.


In [ ]:
def _prepare_vendor_input(vendor_details):
    """Convert one vendor's raw values into the same encoded/scaled feature format used by the model."""
    row = pd.DataFrame([vendor_details])

    # Use the same categorical encoders learned in load_data().
    for col, le in label_encoders.items():
        if col not in row.columns:
            raise ValueError(f"Missing required categorical field: {col}")
        value = row.loc[0, col]
        if value not in le.classes_:
            raise ValueError(
                f"Unknown value '{value}' for {col}. Allowed values include: {list(le.classes_)}"
            )
        row[col] = le.transform([value])[0]

    # Match the training feature order exactly.
    row = row[X.columns]

    # Fill any missing numeric input with the training-data median.
    row = row.fillna(X.median(numeric_only=True))

    return sc.transform(row)


def predict_vendor(vendor_details):
    """Predict survival for one new street-food vendor."""
    X_new = _prepare_vendor_input(vendor_details)
    prediction = int(best_model.predict(X_new)[0])

    if hasattr(best_model, 'predict_proba'):
        survival_probability = float(best_model.predict_proba(X_new)[0, 1])
    elif hasattr(best_model, 'decision_function'):
        raw = float(best_model.decision_function(X_new)[0])
        survival_probability = 1 / (1 + np.exp(-raw))
    else:
        survival_probability = float(prediction)

    if survival_probability >= 0.70:
        risk_level = 'LOW'
    elif survival_probability >= 0.50:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'HIGH'

    result = {
        'Prediction': 'SURVIVED' if prediction == 1 else 'DID NOT SURVIVE',
        'Survival Probability': round(survival_probability * 100, 2),
        'Risk Level': risk_level,
        'Best Model': best_name
    }

    print('\n' + '=' * 45)
    print('VENDOR SURVIVAL RESULT')
    print('=' * 45)
    print('Prediction         :', result['Prediction'])
    print('Survival Probability:', f"{result['Survival Probability']:.2f}%")
    print('Risk Level         :', result['Risk Level'])
    print('Best Model         :', result['Best Model'])
    print('=' * 45)

    if 'feature_importance' in globals() and not feature_importance.empty:
        print('Top influential factors:')
        for f in feature_importance.head(5)['Feature']:
            print('-', f)

    return result


In [ ]:
# Example: replace these values with the details of a real/new vendor.
# The categorical examples below use the first valid category learned from your dataset.
# Do not include vendor_id or vendor_survived here.
example_vendor = {
    'city': label_encoders['city'].classes_[0],
    'zone_type': label_encoders['zone_type'].classes_[0],
    'vendor_age_years': 35,
    'years_in_business': 8,
    'food_category': label_encoders['food_category'].classes_[0],
    'license_status': label_encoders['license_status'].classes_[0],
    'avg_daily_revenue_inr': 3500,
    'avg_daily_customers': 120,
    'monthly_stall_rent_inr': 12000,
    'num_helpers': 2,
    'hours_open_per_day': 10,
    'competition_within_100m': 4,
    'monthly_health_inspection_score': 85,
    'had_fine_last_year': 0,
    'avg_monthly_rainfall_mm': 90,
    'season_of_observation': label_encoders['season_of_observation'].classes_[0],
    'has_online_presence': 1,
    'customer_complaint_rate': 0.03
}

# Run this after replacing the numeric values with the vendor's real details.
prediction_result = predict_vendor(example_vendor)


## Batch Prediction for the Full CSV
This optional section predicts every vendor in a CSV and creates a new file containing the model's survival prediction and probability.


In [ ]:
def predict_all_vendors(input_csv, output_csv='vendor_survival_predictions.csv'):
    raw = pd.read_csv(input_csv)
    vendor_ids = raw['vendor_id'].copy() if 'vendor_id' in raw.columns else pd.Series(range(len(raw)))

    data = raw.drop(columns=['vendor_id', 'vendor_survived'], errors='ignore').copy()

    for col, le in label_encoders.items():
        if col in data.columns:
            data[col] = data[col].fillna(le.classes_[0])
            unknown = ~data[col].isin(le.classes_)
            if unknown.any():
                raise ValueError(f"Unknown category values found in column '{col}': {data.loc[unknown, col].unique()}")
            data[col] = le.transform(data[col])

    for col in X.columns:
        if col not in data.columns:
            raise ValueError(f"Missing required feature column: {col}")

    data = data[X.columns]
    data = data.fillna(X.median(numeric_only=True))
    data_scaled = sc.transform(data)

    predictions = best_model.predict(data_scaled)
    if hasattr(best_model, 'predict_proba'):
        probabilities = best_model.predict_proba(data_scaled)[:, 1]
    elif hasattr(best_model, 'decision_function'):
        raw_score = best_model.decision_function(data_scaled)
        probabilities = 1 / (1 + np.exp(-raw_score))
    else:
        probabilities = predictions.astype(float)

    result_df = pd.DataFrame({
        'vendor_id': vendor_ids,
        'prediction': np.where(predictions == 1, 'SURVIVED', 'DID NOT SURVIVE'),
        'survival_probability': np.round(probabilities * 100, 2)
    })
    result_df['risk_level'] = pd.cut(
        probabilities,
        bins=[-np.inf, 0.50, 0.70, np.inf],
        labels=['HIGH', 'MEDIUM', 'LOW']
    )
    result_df.to_csv(output_csv, index=False)
    print(f'Saved {len(result_df)} predictions to: {output_csv}')
    return result_df

# Example:
# all_predictions = predict_all_vendors(path)
# display(all_predictions.head())
